# QM640 · 03 Time-based split, 385-ticket test sets and I1 baselines (RQ1 preliminary)
Runs on ServiceNow and Rabobank outputs of notebook 01. CPU runtime is enough (about 3 minutes). *Runtime → Run all*, then tell Claude "step 3 done".

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')

## setup (local paths are replaced by Drive paths in Colab)

In [ ]:
import pandas as pd, numpy as np, json, time, datetime, warnings
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import OrdinalEncoder
from statsmodels.stats.proportion import proportion_confint
import xgboost as xgb
warnings.filterwarnings('ignore')
RAW, PROCESSED, RESULTS = DATA_ROOT/'raw', DATA_ROOT/'processed', DATA_ROOT/'results'
FIG = RESULTS/'figures'; FIG.mkdir(parents=True, exist_ok=True)
SEED = 640; N_TEST = 385

## ServiceNow - creation-time features (first event) and final labels (last event)

In [ ]:
import zipfile, io
zp = next((RAW/'servicenow').glob('*.zip'), None) or next(RAW.glob('incident*.zip'))
with zipfile.ZipFile(zp) as z:
    name = [n for n in z.namelist() if n.endswith('.csv')][0]
    ev = pd.read_csv(z.open(name), na_values=['?'], low_memory=False)
ev['opened_at'] = pd.to_datetime(ev['opened_at'], dayfirst=True, errors='coerce')
ev = ev.drop_duplicates().sort_values(['number', 'sys_mod_count'])
first, last = ev.groupby('number').head(1).set_index('number'), ev.groupby('number').tail(1).set_index('number')
FEATS = ['contact_type', 'location', 'u_symptom', 'impact', 'urgency', 'priority', 'opened_by', 'caller_id']
sn = first[FEATS + ['opened_at']].copy()
sn['first_category'] = first['category']; sn['first_group'] = first['assignment_group']
sn['subcategory_first'] = first['subcategory']
sn['category'] = last['category']; sn['assignment_group'] = last['assignment_group']
sn['u_priority_confirmation'] = last['u_priority_confirmation']
sn = sn.reset_index()
# human-correction signal (RQ4): label changed between creation and closure
corr = {'category_changed_%': round(100*(sn['first_category'].notna() & (sn['first_category'] != sn['category'])).mean(), 2),
        'group_changed_%': round(100*(sn['first_group'].notna() & (sn['first_group'] != sn['assignment_group'])).mean(), 2),
        'group_missing_at_creation_%': round(100*sn['first_group'].isna().mean(), 2)}
print('ServiceNow incidents:', len(sn), corr)

## time-based split and the 385-ticket test set

In [ ]:
def time_split(df, tcol, test_frac=0.2, val_frac=0.1):
    df = df.dropna(subset=[tcol]).sort_values(tcol).reset_index(drop=True)
    n = len(df); a, b = int(n*(1-test_frac-val_frac)), int(n*(1-test_frac))
    return df.iloc[:a], df.iloc[a:b], df.iloc[b:]
def stratified_sample(df, label, n, seed=SEED):
    df = df.dropna(subset=[label])
    w = df[label].map(df[label].value_counts(normalize=True))
    return df.sample(n=min(n, len(df)), weights=w*0+1, random_state=seed)  # simple random within recent period
def wilson(k, n):
    lo, hi = proportion_confint(k, n, alpha=.05, method='wilson'); return round(100*lo, 1), round(100*hi, 1)

def run_task(df, feats, label, task, dataset, tcol):
    d = df.dropna(subset=[label]).copy()
    # keep classes with enough history; rare classes grouped as "other" for training only
    tr, va, te_pool = time_split(d, tcol)
    test = te_pool.sample(n=min(N_TEST, len(te_pool)), random_state=SEED)
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)
    Xtr = enc.fit_transform(tr[feats].astype(str)); Xte = enc.transform(test[feats].astype(str))
    classes = tr[label].value_counts().index.tolist(); cmap = {c: i for i, c in enumerate(classes)}
    ytr = tr[label].map(cmap).values; yte_lbl = test[label].values
    out = []
    # Majority-class baseline
    maj = classes[0]; pred = np.array([maj]*len(test))
    out.append(('Majority class', pred, 0.0))
    # Random Forest
    t0 = time.time(); rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED, class_weight='balanced_subsample').fit(Xtr, ytr)
    pred = np.array(classes)[rf.predict(Xte)]; out.append(('I1 Random Forest', pred, time.time()-t0))
    # XGBoost (classes absent from training cannot be predicted)
    t0 = time.time()
    xg = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=.9, colsample_bytree=.9, tree_method='hist', random_state=SEED, n_jobs=-1)
    xg.fit(Xtr, ytr); pred = np.array(classes)[xg.predict(Xte)]; out.append(('I1 XGBoost', pred, time.time()-t0))
    rows = []
    for model, pred, secs in out:
        k = int((pred == yte_lbl).sum()); n = len(test)
        rows.append({'Dataset': dataset, 'Decision': task, 'Model': model, 'Train rows': len(tr), 'Test rows': n,
                     'Classes': d[label].nunique(), 'Accuracy %': round(100*k/n, 1), '95% CI (Wilson)': f'{wilson(k, n)[0]}–{wilson(k, n)[1]}',
                     'Macro-F1': round(f1_score(yte_lbl, pred, average='macro', zero_division=0), 3),
                     'Train+predict s': round(secs, 1), 'Unseen test classes %': round(100*(~pd.Series(yte_lbl).isin(classes)).mean(), 1)})
    return rows, test, {m: p for m, p, _ in out}

res, tests, preds = [], {}, {}
r, t, p = run_task(sn, FEATS, 'category', 'D1 classify (category)', 'ServiceNow', 'opened_at'); res += r; tests['sn_D1'] = t; preds['sn_D1'] = p
r, t, p = run_task(sn, FEATS + ['first_category', 'subcategory_first'], 'assignment_group', 'D5 assign team', 'ServiceNow', 'opened_at'); res += r; tests['sn_D5'] = t; preds['sn_D5'] = p

## Rabobank - D5 (final team from activity log) and D4 proxy (closure code)

In [ ]:
inc = pd.read_parquet(PROCESSED/'rabobank_incidents.parquet')
import sys
act = pd.read_csv(RAW/'bpi2014'/'Detail_Incident_Activity.csv', sep=';', low_memory=False)
act.columns = [c.strip().lower().replace(' ', '_') for c in act.columns]
act['datestamp'] = pd.to_datetime(act['datestamp'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
act = act.sort_values(['incident_id', 'datestamp'])
grp = act.dropna(subset=['assignment_group']).groupby('incident_id')['assignment_group']
inc = inc.merge(grp.first().rename('first_team'), left_on='incident_id', right_index=True, how='left') \
         .merge(grp.last().rename('final_team'), left_on='incident_id', right_index=True, how='left')
RF_FEATS = ['ci_name_aff', 'ci_type_aff', 'ci_subtype_aff', 'service_component_wbs_aff', 'impact', 'urgency', 'priority', 'category']
inc['open_time'] = pd.to_datetime(inc['open_time'], errors='coerce')
corr['rabobank_team_changed_%'] = round(100*(inc['first_team'].notna() & (inc['first_team'] != inc['final_team'])).mean(), 2)
r, t, p = run_task(inc, RF_FEATS, 'final_team', 'D5 assign team', 'Rabobank', 'open_time'); res += r; tests['rb_D5'] = t; preds['rb_D5'] = p
r, t, p = run_task(inc, RF_FEATS, 'closure_code', 'D4 next action (closure code proxy)', 'Rabobank', 'open_time'); res += r; tests['rb_D4'] = t; preds['rb_D4'] = p
perf = pd.DataFrame(res); perf.to_csv(RESULTS/'rq1_baseline_metrics.csv', index=False)
print(perf.to_string())

## save the test sets (ticket ids) for every later instrument

In [ ]:
for k, t in tests.items():
    idc = 'number' if k.startswith('sn') else 'incident_id'
    t[[idc]].to_csv(PROCESSED/f'testset_{k}.csv', index=False)

## figure - accuracy and macro-F1 by decision and model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
lab = perf['Dataset'] + '\n' + perf['Decision'].str.split(' ').str[0]
for ax, m in zip(axes, ['Accuracy %', 'Macro-F1']):
    piv = perf.assign(task=lab).pivot(index='task', columns='Model', values=m)[['Majority class', 'I1 Random Forest', 'I1 XGBoost']]
    piv.plot.bar(ax=ax, color=['#a0aec0', '#2b6cb0', '#c05621'], rot=0); ax.set_title(m + ' on the held-out test set'); ax.set_xlabel(''); ax.legend(fontsize=8)
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(FIG/'rq1_baseline.png', dpi=200); plt.close()

## run cost of I1 (CPU time priced at an on-demand cloud vCPU rate)

In [ ]:
VCPU_PER_HOUR = 0.048   # USD, typical on-demand cloud vCPU-hour; recorded in configs/prices.yaml
cost = perf[perf['Model'] != 'Majority class'].assign(**{'USD per 1,000 tickets (train+predict, upper bound)':
        lambda d: (d['Train+predict s']/3600*VCPU_PER_HOUR*4/(d['Test rows'])*1000).round(5)})
cost.to_csv(RESULTS/'rq1_baseline_cost.csv', index=False)

## step summary

In [ ]:
def mdt(df):
    try: return df.to_markdown(index=False)
    except Exception: return df.to_string()
lines = [f'# Step 03 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', '', '## Human-correction signals (RQ4)', json.dumps(corr), '',
         '## RQ1 preliminary: I1 baselines on the held-out test sets', mdt(perf), '', '## Test sets', json.dumps({k: len(t) for k, t in tests.items()}),
         '', '## Cost', mdt(cost[['Dataset', 'Decision', 'Model', 'Train+predict s', 'USD per 1,000 tickets (train+predict, upper bound)']])]
(RESULTS/'step03_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))

## RQ2 preliminary - risk-coverage of the XGBoost confidence score

In [ ]:
def risk_coverage(df, feats, label, tcol, name):
    d = df.dropna(subset=[label]).copy()
    tr, va, te_pool = time_split(d, tcol)
    test = te_pool.sample(n=min(N_TEST, len(te_pool)), random_state=SEED)
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)
    Xtr = enc.fit_transform(tr[feats].astype(str)); Xte = enc.transform(test[feats].astype(str))
    classes = tr[label].value_counts().index.tolist(); cmap = {c: i for i, c in enumerate(classes)}
    m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=.9, colsample_bytree=.9, tree_method='hist', random_state=SEED, n_jobs=-1)
    m.fit(Xtr, tr[label].map(cmap).values)
    pr = m.predict_proba(Xte); conf = pr.max(1); pred = np.array(classes)[pr.argmax(1)]
    ok = (pred == test[label].values)
    o = np.argsort(-conf); cov = np.arange(1, len(o)+1)/len(o); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1)
    def cov_at(p): 
        idx = np.where(prec >= p)[0]; return round(100*cov[idx.max()], 1) if len(idx) else 0.0
    # expected calibration error (10 bins)
    bins = np.linspace(0, 1, 11); ece = 0
    for a, b in zip(bins[:-1], bins[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): ece += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return {'Dataset/decision': name, 'Accuracy % (all)': round(100*ok.mean(), 1), 'Coverage % at 98% precision (automate)': cov_at(.98),
            'Coverage % at 90% precision (recommend)': cov_at(.90), 'ECE': round(ece, 3)}, (cov, prec)
rc_rows, curves = [], {}
for nm, (df_, f_, l_, t_) in {'ServiceNow D5': (sn, FEATS + ['first_category', 'subcategory_first'], 'assignment_group', 'opened_at'),
                              'ServiceNow D1': (sn, FEATS, 'category', 'opened_at'),
                              'Rabobank D5': (inc, RF_FEATS, 'final_team', 'open_time'),
                              'Rabobank D4': (inc, RF_FEATS, 'closure_code', 'open_time')}.items():
    r, c = risk_coverage(df_, f_, l_, t_, nm); rc_rows.append(r); curves[nm] = c
rc = pd.DataFrame(rc_rows); rc.to_csv(RESULTS/'rq2_risk_coverage_prelim.csv', index=False); print(rc.to_string())
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for nm, (cov, prec) in curves.items(): ax.plot(100*cov, 100*prec, label=nm)
ax.axhline(98, ls='--', c='grey', lw=.8); ax.axhline(90, ls=':', c='grey', lw=.8)
ax.text(1, 98.5, '98% automate', fontsize=8, color='grey'); ax.text(1, 90.5, '90% recommend', fontsize=8, color='grey')
ax.set_xlabel('Coverage: % of tickets decided (most confident first)'); ax.set_ylabel('Precision %'); ax.set_ylim(0, 102)
ax.set_title('Risk–coverage of I1 (XGBoost) confidence'); ax.legend(fontsize=8); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(FIG/'rq2_risk_coverage_prelim.png', dpi=200); plt.close()

## RQ4 preview - rules compiled from history (support >= 20, purity >= 98% / 90%)

In [ ]:
def rule_preview(df, key, label, tcol, name, min_support=20):
    d = df.dropna(subset=[label]).copy()
    tr, va, te_pool = time_split(d, tcol)
    test = te_pool.sample(n=min(N_TEST, len(te_pool)), random_state=SEED)
    k = lambda x: x[key].fillna('NA').astype(str).agg('|'.join, axis=1)
    tr = tr.assign(_k=k(tr)); test = test.assign(_k=k(test))
    stats = tr.groupby('_k')[label].agg(n='size', top=lambda s: s.value_counts().index[0], purity=lambda s: s.value_counts(normalize=True).iloc[0])
    out = {'Dataset/decision': name, 'Rule key': ' + '.join(key)}
    for thr, tag in [(.98, 'automate'), (.90, 'recommend')]:
        rules = stats[(stats['n'] >= min_support) & (stats['purity'] >= thr)]
        hit = test['_k'].isin(rules.index)
        correct = (test.loc[hit, '_k'].map(rules['top']) == test.loc[hit, label]).mean() if hit.any() else np.nan
        out[f'Rules ({tag})'] = len(rules); out[f'Test coverage % ({tag})'] = round(100*hit.mean(), 1)
        out[f'Test precision % ({tag})'] = round(100*correct, 1) if hit.any() else None
    return out
rp = pd.DataFrame([
    rule_preview(sn, ['first_category', 'subcategory_first', 'u_symptom'], 'assignment_group', 'opened_at', 'ServiceNow D5'),
    rule_preview(inc, ['ci_name_aff', 'service_component_wbs_aff'], 'final_team', 'open_time', 'Rabobank D5'),
    rule_preview(inc, ['ci_name_aff', 'service_component_wbs_aff'], 'closure_code', 'open_time', 'Rabobank D4')])
rp.to_csv(RESULTS/'rq4_rule_preview.csv', index=False); print(rp.to_string())
with open(RESULTS/'step03_summary.md', 'a') as fh:
    fh.write('\n\n## RQ2 preliminary: risk-coverage (XGBoost)\n' + mdt(rc) + '\n\n## RQ4 preview: compiled rules from history\n' + mdt(rp))